<a href="https://colab.research.google.com/github/GinnaGomez09/proyecto_aplicado_javeriana/blob/main/notebooks/02_reduccion_dataset_recetas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# ============================================================
# Clonar repositorio y preparar entorno
# ============================================================

!git clone https://github.com/GinnaGomez09/proyecto_aplicado_javeriana.git
%cd proyecto_aplicado_javeriana

Cloning into 'proyecto_aplicado_javeriana'...
remote: Enumerating objects: 312, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 312 (delta 112), reused 18 (delta 18), pack-reused 142 (from 1)
Receiving objects: 100% (312/312), 3.72 MiB | 5.58 MiB/s, done.
Resolving deltas: 100% (166/166), done.
/content/proyecto_aplicado_javeriana/proyecto_aplicado_javeriana/proyecto_aplicado_javeriana


In [15]:
# ============================================================
# 02_REDUCCION_DATASET.ipynb
# Reducción y depuración del dataset antes del pipeline NLP
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Cargar dataset original
# ------------------------------------------------------------

input_path = "data/raw/recetas/recetas_ingredientes.csv"

recetas = pd.read_csv(input_path)

print("Dimensiones originales:", recetas.shape)
print("Número original de recetas:", recetas["receta_uuid"].nunique())

Dimensiones originales: (31587, 16)
Número original de recetas: 26951


In [16]:
# ------------------------------------------------------------
# 3. Identificar dominio/fuente de las recetas
# ------------------------------------------------------------

recetas_reducidas = recetas.copy()

recetas_reducidas["fuente"] = (
    recetas_reducidas["receta_url"]
    .astype(str)
    .str.extract(r"https?://([^/]+)")
)

print("Distribución de fuentes:")
display(
    recetas_reducidas["fuente"]
    .value_counts()
)

Distribución de fuentes:


,count
fuente,
www.recetasgratis.net,20908
www.recetas.com,5597
www.mycolombianrecipes.com,3796
www.recetasnestle.com.co,1286


In [17]:
# ------------------------------------------------------------
# 4. Conservar únicamente recetas mejor estructuradas
# ------------------------------------------------------------
# Se conservan recetas de:
# www.mycolombianrecipes.com
#
# Razones:
# - Mejor estructura ingrediente-por-fila
# - Menor cantidad de ruido textual
# - Recetas colombianas relevantes
# - Más fáciles de procesar con recursos limitados
# ------------------------------------------------------------

recetas_reducidas = recetas_reducidas[
    recetas_reducidas["fuente"] == "www.mycolombianrecipes.com"
].copy()

print("Dimensiones después del filtrado por fuente:", recetas_reducidas.shape)
print("Número de recetas:", recetas_reducidas["receta_uuid"].nunique())

Dimensiones después del filtrado por fuente: (3796, 17)
Número de recetas: 446


In [18]:
# ------------------------------------------------------------
# 5. Eliminar registros problemáticos
# ------------------------------------------------------------

# Eliminar nulos importantes
recetas_reducidas = recetas_reducidas.dropna(
    subset=[
        "receta_uuid",
        "receta_titulo",
        "ingrediente_linea",
        "ingrediente_nombre"
    ]
).copy()

# Eliminar duplicados completos
recetas_reducidas = recetas_reducidas.drop_duplicates().copy()

# Eliminar líneas excesivamente largas
# (normalmente corresponden a ingredientes concatenados
# o recetas mal estructuradas)

recetas_reducidas = recetas_reducidas[
    recetas_reducidas["ingrediente_linea"]
    .astype(str)
    .str.len() <= 180
].copy()

print("Dimensiones después de limpieza:", recetas_reducidas.shape)
print("Número de recetas:", recetas_reducidas["receta_uuid"].nunique())

Dimensiones después de limpieza: (3795, 17)
Número de recetas: 446


In [19]:
# ------------------------------------------------------------
# 6. Conservar recetas con cantidad razonable de ingredientes
# ------------------------------------------------------------
# Se eliminan:
# - recetas demasiado pequeñas
# - recetas extremadamente largas/problemáticas
# ------------------------------------------------------------

conteo_ingredientes = (
    recetas_reducidas
    .groupby("receta_uuid")
    .size()
    .reset_index(name="n_ingredientes")
)

display(conteo_ingredientes.describe())

# Mantener recetas entre 3 y 25 ingredientes

recetas_validas = conteo_ingredientes[
    conteo_ingredientes["n_ingredientes"].between(3, 25)
]["receta_uuid"]

recetas_reducidas = recetas_reducidas[
    recetas_reducidas["receta_uuid"].isin(recetas_validas)
].copy()

print("Dimensiones finales:", recetas_reducidas.shape)
print("Número final de recetas:", recetas_reducidas["receta_uuid"].nunique())

,n_ingredientes
count,446.000000
mean,8.508969
std,4.203119
min,1.000000
25%,5.000000
50%,8.000000
75%,11.000000
max,24.000000


Dimensiones finales: (3777, 17)
Número final de recetas: 436


In [20]:
# ------------------------------------------------------------
# 7. Eliminar columnas innecesarias
# ------------------------------------------------------------

recetas_reducidas = recetas_reducidas.drop(
    columns=[
        "receta_url",
        "fuente"
    ]
)

print("Columnas finales:")
print(recetas_reducidas.columns.tolist())

Columnas finales:
['receta_uuid', 'receta_titulo', 'ingrediente_id', 'ingrediente_linea', 'cantidad_original', 'cantidad_conv', 'cantidad_min', 'cantidad_max', 'unidad', 'ingrediente_nombre', 'tcac_alimento_codigo', 'tcac_alimento_nombre', 'match_method', 'match_score', 'cantidad_gramos_est']


In [21]:
# ------------------------------------------------------------
# 8. Vista final del dataset reducido
# ------------------------------------------------------------

recetas_reducidas.head()

,receta_uuid,receta_titulo,ingrediente_id,ingrediente_linea,cantidad_original,cantidad_conv,cantidad_min,cantidad_max,unidad,ingrediente_nombre,tcac_alimento_codigo,tcac_alimento_nombre,match_method,match_score,cantidad_gramos_est
0,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,1,1 taza de harina de arepa blanca o amarilla,1,1.000000,NaN,NaN,taza,harina de arepa blanca o amarilla,NaN,NaN,NaN,NaN,NaN
1,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,2,1 taza de agua tibia,1,1.000000,NaN,NaN,taza,agua tibia,NaN,NaN,NaN,NaN,NaN
2,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,3,⅓ taza de queso mozzarella o queso blanco rallado,⅓,0.333333,NaN,NaN,taza,queso mozzarella o queso blanco rallado,NaN,NaN,NaN,NaN,NaN
3,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,4,2 cucharadas de mantequilla,2,2.000000,NaN,NaN,cucharadas,mantequilla,NaN,NaN,NaN,NaN,NaN
4,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,5,Sal,NaN,NaN,NaN,NaN,NaN,Sal,NaN,NaN,NaN,NaN,NaN


In [22]:
# ------------------------------------------------------------
# 9. Guardar dataset reducido
# ------------------------------------------------------------

Path("data/interim").mkdir(parents=True, exist_ok=True)

output_path = "data/interim/recetas_reducidas.csv"

recetas_reducidas.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset reducido guardado en:")
print(output_path)

print("\nResumen final:")
print("Filas:", recetas_reducidas.shape[0])
print("Recetas:", recetas_reducidas["receta_uuid"].nunique())

Dataset reducido guardado en:
data/interim/recetas_reducidas.csv

Resumen final:
Filas: 3777
Recetas: 436


In [23]:
# ============================================================
# Descargar archivo CSV
# ============================================================

from google.colab import files

files.download("data/interim/recetas_reducidas.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Reducción y depuración del dataset

Antes de iniciar las etapas avanzadas del pipeline NLP, se realiza una reducción y depuración del dataset original con el fin de:

- Disminuir el ruido textual.
- Eliminar columnas irrelevantes para el proyecto.
- Conservar únicamente recetas con mejor estructura.
- Reducir el costo computacional.
- Facilitar la validación y desarrollo del pipeline.

Se seleccionan únicamente recetas provenientes de fuentes con estructura ingrediente-por-fila, eliminando registros problemáticos, recetas excesivamente largas y columnas vacías relacionadas con la integración futura con la TCAC.